In [13]:
# Library Imports.
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn import metrics

# Allows plots to appear directly in the notebook.
%matplotlib inline


In [14]:
# Reading from a csv file, into a data frame
df_train = pd.read_csv('ppr-group-25204989-train-Formodeling.csv', keep_default_na=True, delimiter=',', skipinitialspace=True)
# Show data frame first few rows
df_train.head(10)

,Price(€),NotFullMarketPrice,VATExclusive,DescriptionofProperty,SaleYear,SaleMonthIndex,Region_Encoded
0,62500.0,0,0,0,2016,7,140133.584455
1,150000.0,1,0,0,2016,12,198460.380569
2,94000.0,0,0,0,2016,8,140133.584455
3,149500.0,0,1,1,2016,2,78111.708000
4,170000.0,0,0,0,2016,10,101768.538462
5,50000.0,0,0,0,2016,12,169894.251731
6,15000.0,0,0,0,2016,10,486409.449362
7,160000.0,0,0,0,2016,8,139912.414348
8,320000.0,0,1,1,2016,12,178090.236923
9,225000.0,0,0,0,2016,5,357260.467719


In [15]:
# Reading from a csv file, into a data frame
df_test = pd.read_csv('ppr-group-25204989-test-Forevaluation.csv', keep_default_na=True, delimiter=',', skipinitialspace=True)
# Show data frame first few rows
df_test.head(10)

,Price(€),NotFullMarketPrice,VATExclusive,DescriptionofProperty,SaleYear,SaleMonthIndex,Region_Encoded
0,2525000.00,0,0,0,2025,120,543378.937302
1,260000.00,0,0,0,2025,118,229921.623040
2,361233.00,0,1,1,2025,120,299132.044691
3,435000.00,0,0,0,2025,114,412896.226415
4,334000.00,0,0,0,2025,119,285224.814920
5,390000.00,0,0,0,2025,119,354279.664683
6,845814.97,0,1,1,2025,117,466855.762541
7,282000.00,0,0,0,2025,109,266956.674667
8,550000.00,0,0,0,2025,112,320858.648413
9,400000.00,0,0,0,2025,118,332809.963642


In [16]:
# This function is used repeatedly to compute, print and return the regression metrics
def getMetrics(testActualVal, predictions):
    mae=metrics.mean_absolute_error(testActualVal, predictions)
    rmse=metrics.mean_squared_error(testActualVal, predictions)**0.5
    r2=metrics.r2_score(testActualVal, predictions)
    print('\n==============================================================================')
    print("MAE: ", mae)
    #print("MSE: ", metrics.mean_squared_error(testActualVal, predictions))
    print("RMSE: ", rmse)
    print("R2: ", r2)
    return (mae,rmse,r2)

In [19]:
# Define descriptive features and target feature
features = ['NotFullMarketPrice', 'VATExclusive', 'SaleMonthIndex', 'Region_Encoded']
target = ['Price(€)']

In [21]:
X_train_unscaled = df_train[features]
y_train = df_train[target]

X_test_unscaled = df_test[features]
y_test = df_test[target]


In [22]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

num_feature=['SaleMonthIndex','Region_Encoded']
binary_features=['NotFullMarketPrice', 'VATExclusive']

preprocess_scaled = ColumnTransformer(
    [
        ('binary', 'passthrough', binary_features),
        ('num', StandardScaler(), num_feature),
    ]
 )
    # add the code to scale here
    # fit on training data
preprocess_scaled.fit(X_train_unscaled)

# transform both train and validation
X_train = preprocess_scaled.transform(X_train_unscaled)
X_test = preprocess_scaled.transform(X_test_unscaled)

# train the model
ridge = RidgeCV()
ridge.fit(X_train, y_train)

# Print the weights learned for each feature.
print("\nIntercept: \n", ridge.intercept_)
print("Features and coeficients:", list(zip(features, ridge.coef_))[:10])

# Predicted price on validation set
pred = ridge.predict(X_test)
(mae,rmse,r2)=getMetrics(y_test, pred)

feature_importance = pd.DataFrame({'feature': features, 'importance':ridge.coef_})
display(feature_importance.sort_values('importance', ascending=False))


Intercept: 
 [280046.15454299]
Features and coeficients: [('NotFullMarketPrice', np.float64(-99839.0612907313)), ('VATExclusive', np.float64(27060.000643164996)), ('SaleMonthIndex', np.float64(5265.630665447505)), ('Region_Encoded', np.float64(101895.91027975545))]

MAE:  150409.9747395143
RMSE:  672381.3190735974
R2:  0.030926974373742033


,feature,importance
3,Region_Encoded,101895.910280
1,VATExclusive,27060.000643
2,SaleMonthIndex,5265.630665
0,NotFullMarketPrice,-99839.061291
